# ViSceT5 — Finetune **MRA** (Mixture-of-Resolution Adaptation) @512 · Colab **A100**

Nhánh `exp/mra`. Runtime → **A100 GPU** (High-RAM). Chạy tuần tự từ trên xuống.

**MRA:** ViT giữ **224px**; nhánh CNN (ConvNeXtV2) nhận **512px**, lấy 3 stage cuối
(64×64, 32×32, 16×16) rồi qua **lớp align** `adaptive_avg_pool2d → 14×14`, bơm vào 3 tầng
ViT cuối qua MR-Adapter — token ảnh vẫn **196**.

**Công bằng ablation:** mặc định giữ đúng cấu hình baseline (batch 4 × accum 2 = *effective* 8,
fp32) để so trực tiếp với baseline/qaclip/ocr. A100 chỉ dùng để **tăng tốc** qua **TF32**
(gần như không đổi số học) và eval batch lớn hơn. Muốn nhanh gấp đôi thì bật *fast mode*
(bf16) ở cell 3 — nhưng đó là đổi precision so với baseline fp32.

### 1. Clone repo + checkout nhánh `exp/mra`

In [ ]:
!git clone https://github.com/Kussssssss/ViSceT5.git /content/ViSceT5
%cd /content/ViSceT5
!git fetch origin exp/mra && git checkout exp/mra && git pull
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

### 2. Cài dependencies

In [ ]:
!bash setup.sh
# Nếu Colab báo cần restart: Runtime → Restart session, rồi chạy tiếp TỪ cell 3.

### 3. Cấu hình ablation MRA + A100 (đặt qua env — không sửa YAML)

`run_pipeline.py` đọc các biến này và tự thêm cờ CLI cho `finetune.py`.

In [ ]:
%env CUDA_VISIBLE_DEVICES=0
%env STAGE=finetune
# --- Ablation: chỉ bật MRA (tắt Visual Search để không dùng ConvNeXt hai lần) ---
%env ABLATION_USE_QACLIP=true
%env ABLATION_USE_VS=false
%env ABLATION_USE_OCR=true
%env ABLATION_USE_MRA=true
%env MRA_HIGH_RES=512
# --- A100: tăng tốc, GIỮ nguyên effective-batch để công bằng với baseline ---
%env TF32=true                       # A100 TensorFloat-32: nhanh, ~fp32
%env PER_DEVICE_EVAL_BATCH_SIZE=16   # eval nhanh hơn (không ảnh hưởng train)
%env DATALOADER_NUM_WORKERS=8
# (train batch 4 × accum 2 = effective 8, fp32 — theo configs/finetune.yaml)

# --- FAST MODE (tuỳ chọn): nhanh ~2x nhưng ĐỔI precision so với baseline fp32 ---
#%env BF16=true
#%env PER_DEVICE_TRAIN_BATCH_SIZE=8
#%env GRADIENT_ACCUMULATION_STEPS=1   # giữ effective-batch = 8

# (Tuỳ chọn) upload model finetune lên HF khi xong: điền cả 2 dòng.
import os
os.environ['HF_TOKEN'] = ''   # hf_xxx
os.environ['HF_REPO']  = ''   # vd: Kus669/ViSceT5-mra

### 4. Pre-flight — kiểm tra MRA dựng đúng (không cần dataset)

Dựng model với MRA, forward+backward trên dữ liệu ngẫu nhiên: kỳ vọng token ảnh **196**,
cổng `g ≈ 0` lúc khởi tạo, gradient chảy vào ConvNeXt + MR-Adapter, không có grad non-finite.

In [ ]:
import os, numpy as np, torch
from PIL import Image
from configs.model_config import OpenViVQAConfig
from models.openvivqa_model import OpenViVQAModel

torch.manual_seed(0); np.random.seed(0)
cfg = OpenViVQAConfig()
cfg.pretrain = False
cfg.ablation_use_qaclip = True; cfg.ablation_use_vs = False
cfg.ablation_use_ocr = True;   cfg.ablation_use_mra = True
cfg.mra_high_res = 512

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
m = OpenViVQAModel(cfg).to(dev)
enc = m.qa_clip.vision_model.encoder
print('mra_high_res :', m.mra_high_res)
print('layer_ids    :', enc.mra_layer_ids, '| adapters d_cnn:', [a.fh.in_features for a in enc.mra_adapters])

B, P, DD = 2, 3, 256
imgs = [Image.fromarray((np.random.rand(600,1000,3)*255).astype('uint8')) for _ in range(B)]
pv = m.image_processor(images=imgs, return_tensors='pt')['pixel_values'].to(dev)

m.use_mra = True
hi = m._encode_mra_highres(imgs, torch.device(dev))
print('hi feats (sau align):', [tuple(h.shape) for h in hi], '-> phai la (2,196,C)')

batch = dict(
    input_ids=torch.randint(5,1000,(B,16)).to(dev),
    attention_mask=torch.cat([torch.ones(B,9),torch.zeros(B,7)],1).long().to(dev),
    labels=torch.randint(5,1000,(B,8)).to(dev), pixel_values=pv, pil_images=imgs,
    ocr_info=[{'width':1000.,'height':600.,'boxes_word_all':torch.rand(P,4).sort(-1)[0],
               'word_mask_all':torch.ones(P,dtype=torch.long),
               'det_features':torch.randn(P,DD),'rec_features':torch.randn(P,DD)} for _ in range(B)],
    twa_word_ids=torch.randint(5,1000,(B,P*2)).to(dev),
    ocr_to_word_map=torch.arange(P*2).remainder(P).unsqueeze(0).repeat(B,1).to(dev),
    twa_ocr_char=torch.randint(0,100,(B,P,50)).to(dev), twa_ocr_char_mask=torch.ones(B,P,50).to(dev),
    ocr_mask_box=torch.ones(B,P,dtype=torch.long).to(dev))

m.train(); out = m(**batch)
print('loss =', round(float(out['loss']),4), '| finite:', bool(torch.isfinite(out['loss'])),
      '| logits:', tuple(out['logits'].shape))
out['loss'].backward()
gs = lambda pr: sum(float(p.grad.abs().sum()) for n,p in m.named_parameters() if pr(n) and p.grad is not None)
print('|grad| ConvNeXt hi =', round(gs(lambda n:'mra_cnn' in n or 'visual_search.cnn' in n),2))
print('|grad| MR-Adapter   =', round(gs(lambda n:'mra_adapters' in n),2))
bad = [n for n,p in m.named_parameters() if p.grad is not None and not torch.isfinite(p.grad).all()]
print('grad non-finite    :', len(bad), '(phai = 0)')
del m, out, batch, pv, hi
import gc; gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('\nPRE-FLIGHT OK' if not bad else '\nPRE-FLIGHT FAIL')

### 5. Smoke test (dataset nhỏ, ít step) — bắt lỗi wiring trước khi chạy full

`MOCK_TEST=true` chỉ áp cho lệnh này; các env ở cell 3 vẫn giữ nguyên.

In [ ]:
!MOCK_TEST=true python run_pipeline.py

### 6. Finetune đầy đủ trên ViTextVQA

Dùng `configs/finetune.yaml` (5 epoch) + override A100 ở cell 3. Xong sẽ tự upload HF nếu đã
điền `HF_TOKEN`/`HF_REPO`. EM/F1/CIDEr in ở cuối log; model lưu tại `output/finetune`.

In [ ]:
!python run_pipeline.py

### 7. (Tuỳ chọn) Xem nhanh metric cuối

In [ ]:
import json, glob
cands = sorted(glob.glob('output/finetune/**/trainer_state.json', recursive=True) + glob.glob('output/finetune/trainer_state.json'))
if cands:
    st = json.load(open(cands[-1]))
    print('best_metric =', st.get('best_metric'), '| best_ckpt =', st.get('best_model_checkpoint'))
    last = [h for h in st.get('log_history', []) if any(k.startswith('eval_') for k in h)]
    print('eval cuoi:', last[-1] if last else 'N/A')
else:
    print('Chua co trainer_state.json trong output/finetune')